In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("Incremental Flag","0")
increm_flag=dbutils.widgets.get("Incremental Flag")
print(increm_flag)

In [0]:
df_medication_order=spark.sql('''
                     select * from  parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/pharmacy_medication_order`''')

In [0]:
df_medication_order.limit(5).display()

In [0]:
df_dim_patient=spark.sql('SELECT * FROM healthcare.gold.dim_patient')
df_dim_provider=spark.sql('SELECT * FROM healthcare.gold.dim_provider')
df_dim_medication=spark.sql('SELECT * FROM healthcare.gold.dim_medication')
df_fact_encounter=spark.sql('SELECT * FROM healthcare.gold.fact_encounter')
df_dim_date=spark.sql('SELECT * FROM healthcare.gold.dim_date')

In [0]:
df_medication_order.count()

In [0]:
df_staged_fact=df_medication_order.join(df_dim_patient,df_medication_order["mrn"]==df_dim_patient["mrn"],"inner")\
    .join(df_dim_provider,df_medication_order["prescribing_npi"]==df_dim_provider["npi"],"inner")\
        .join(df_dim_medication,df_medication_order["ndc_code"]==df_dim_medication["ndc_code"],"inner")\
                .join(df_fact_encounter,df_medication_order["encounter_id"]==df_fact_encounter["encounter_id"],"inner")\
                    .join(df_dim_date,df_medication_order["order_date"]==df_dim_date["full_date"],"inner")\
                        .select(df_dim_patient["dim_patient_key"],df_dim_provider["dim_provider_key"],df_dim_medication["dim_medication_key"],df_fact_encounter["encounter_key"],df_dim_date["dim_date_key"],df_medication_order["order_id"],df_medication_order["dose"],df_medication_order["frequency"],df_medication_order["order_status"])

In [0]:
df_staged_fact.count()

In [0]:
df_staged_fact.limit(5).display()

In [0]:
if increm_flag=='0':
    max_value=1
else:
    max_value_df=spark.sql('''
            select coalesce(max(medication_key),0) from healthcare.gold.fact_medication_order''')
    max_value=max_value_df.collect()[0][0]+1

In [0]:
if spark.catalog.tableExists('healthcare.gold.fact_medication_order'):
    df_sink=spark.sql('''
        select medication_key,dim_patient_key,
dim_provider_key,
dim_medication_key,
encounter_key,
dim_date_key,
order_id,
dose,
frequency,
order_status
 from healthcare.gold.fact_medication_order''')
else:
    df_sink=spark.sql('''
            select 1 as medication_key,
order_id,
dose,
frequency,
order_status from parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/pharmacy_medication_order` where 1=0''')

In [0]:
df_sink.limit(5).display()

In [0]:
df_filter=df_staged_fact.join(df_sink,df_staged_fact["order_id"]==df_sink["order_id"],"left")\
    .select(df_staged_fact["*"],df_sink["medication_key"])

In [0]:
df_filter.limit(5).display()


In [0]:
df_new_rec=df_filter.filter(df_filter["medication_key"].isNull())
df_existing_rec=df_filter.filter(df_filter["medication_key"].isNotNull())
df_new_rec.limit(5).display()
df_existing_rec.limit(5).display()

In [0]:
dupes = df_existing_rec.groupBy("medication_key").count().filter("count > 1")
dupes.show()

#the duplicates
df_existing_rec.join(dupes.select("medication_key"), "medication_key") \
    .orderBy("medication_key") \
    .display()

In [0]:
windowSpec = Window.orderBy("order_id")
df_insert = df_new_rec.withColumn(
    "medication_key", row_number().over(windowSpec) + lit(max_value - 1)
)

In [0]:
df_insert.limit(5).display()

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("healthcare.gold.fact_medication_order"):
    #insert the new entries
    df_insert.write.format("delta").mode("append").saveAsTable("healthcare.gold.fact_medication_order")
    #update the entries
    deltaTable=DeltaTable.forName(spark,"healthcare.gold.fact_medication_order")
    df_existing_rec_dedup = df_existing_rec.dropDuplicates(["medication_key"])
    deltaTable.alias("target").merge(df_existing_rec_dedup.alias("source"),"target.medication_key = source.medication_key")\
        .whenMatchedUpdate(set={
            "dim_patient_key": "source.dim_patient_key",
            "dim_provider_key": "source.dim_provider_key",
            "dim_medication_key": "source.dim_medication_key",
            "dim_date_key": "source.dim_date_key",
            "encounter_key": "source.encounter_key",
            "order_id": "source.order_id",
            "dose": "source.dose",
            "frequency": "source.frequency",
            "order_status": "source.order_status"
    }).execute()
else:
    df_insert.write.format("delta").mode("append").saveAsTable("healthcare.gold.fact_medication_order")

In [0]:
%sql
select * from healthcare.gold.fact_medication_order